# Figure 3g/3h — MIP vs single-cell aggregates

Per compound and concentration step, the median pairwise Pearson correlation between
replicates in MIP (x) against single-cell aggregates (y). Points above the diagonal
reproduce better in aggregates. 3g colours by concentration step, 3h by pathway;
markers separate the two cell lines.

Numbered `4_` so it runs after `3_PercentReplicating`, which writes the tables it
reads — run_all orders a figure folder alphabetically.

Built from the replicate correlations `3_PercentReplicating` computes and ships as
the Fig 3e/3f source tables, which are the same quantity on both axes.


In [ ]:
# --- repo path bootstrap ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import profiles, SOURCE_DATA_ROOT
from utils.panels import save_panel

import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
sns.set_style('white')

In [ ]:
# Fig 3e/3f source tables: median pairwise Pearson r per (compound, conc step, cell line)
# for MIP and for aggregates. The replicate rows are the ones plotted; the null rows
# belong to 3e/3f only.
def load(panel):
    # Source tables are named for the panel (Fig3e.csv). The older convention appended
    # a slug, so accept both. Raise rather than SystemExit: a notebook treats SystemExit
    # as a clean exit, so this used to "pass" while silently producing no panel.
    direct = SOURCE_DATA_ROOT / f"{panel}.csv"
    hits = [direct] if direct.exists() else sorted(
        pathlib.Path(p) for p in glob.glob(str(SOURCE_DATA_ROOT / f"{panel}_*.csv")))
    if not hits:
        raise FileNotFoundError(
            f"{panel} source table missing — run `python run_all.py --figure Fig3` first, "
            f"which writes it from 3_PercentReplicating")
    d = pd.read_csv(hits[0])
    return d[d["kind"] == "replicate"]

mip = load("Fig3e").rename(columns={"corr": "MIP"})
agg = load("Fig3f").rename(columns={"corr": "Aggregates"})

KEYS = ["Metadata_name", "Metadata_conc_step", "cell_line"]
pairs = mip[KEYS + ["MIP"]].merge(agg[KEYS + ["Aggregates"]], on=KEYS, how="inner")
matched = len(pairs)

# The inner join matches on keys, but a matched row can still carry a missing
# correlation -- 12 do (4 lack MIP only, 8 lack both), because those compound/dose
# combinations had too few surviving replicates to correlate. matplotlib drops them
# silently, so the scatter shows 383 points while the joined frame holds 395. Drop
# them here instead, so the Source Data table is the data behind the graph and its
# row count matches the n quoted in the figure legend.
pairs = pairs.dropna(subset=["MIP", "Aggregates"]).reset_index(drop=True)
print(f"MIP rows {len(mip)}, aggregate rows {len(agg)}, matched {matched}, "
      f"plotted {len(pairs)} (dropped {matched - len(pairs)} with no correlation)")
pairs.head()

In [ ]:
# Pathway per compound, from the profile tables the rest of Figure 3 uses.
pw = []
for cl in ("HCT116", "HT29"):
    d = pd.read_parquet(profiles("exp1_main", f"grit_data_aggregates_{cl}.parquet"),
                        columns=["Metadata_name", "Metadata_pathway"])
    pw.append(d.drop_duplicates())
pathway_of = (pd.concat(pw).dropna().drop_duplicates("Metadata_name")
                .set_index("Metadata_name")["Metadata_pathway"])
pairs["Metadata_pathway"] = pairs["Metadata_name"].map(pathway_of)
print(f"{pairs.Metadata_pathway.isna().sum()} of {len(pairs)} rows without a pathway")

# Same palette and pathway ordering as Fig 4 / Fig 5, so colours agree across figures.
# Shared pathway colours — see utils/palettes.py
from utils.palettes import PATHWAY_COLOURS as LUT, PATHWAY_ORDER as PATHWAYS
MARKERS = {'HCT116': 'o', 'HT29': '+'}

In [ ]:
def agg_vs_mip(hue, palette, panel, caption, legend_title):
    fig, ax = plt.subplots(figsize=(6, 5))
    lo = min(pairs.MIP.min(), pairs.Aggregates.min()) - 0.05
    hi = max(pairs.MIP.max(), pairs.Aggregates.max()) + 0.05
    ax.plot([lo, hi], [lo, hi], ls='--', lw=1, color='black', zorder=1)

    # One call per cell line: seaborn refuses to mix a filled marker with a line-art
    # one inside a single `style` mapping, and the paper uses a dot and a plus.
    for cl, marker in MARKERS.items():
        sub = pairs[pairs.cell_line == cl]
        sns.scatterplot(data=sub, x='MIP', y='Aggregates', hue=hue, palette=palette,
                        marker=marker, s=45, alpha=0.85,
                        linewidth=0 if marker == 'o' else 0.9,
                        edgecolor='none' if marker == 'o' else None,
                        legend=(cl == 'HCT116'), ax=ax, zorder=2)

    # cell-line key, appended to the colour legend
    handles, labels = ax.get_legend_handles_labels()
    for cl, marker in MARKERS.items():
        handles.append(plt.Line2D([], [], ls='', marker=marker, color='0.3', markersize=6))
        labels.append(cl)
    ax.legend(handles, labels, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8,
              title=legend_title, title_fontsize=9, frameon=False)

    ax.set_xlabel('MIP'); ax.set_ylabel('Aggregates')
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    sns.despine()
    save_panel(fig, panel, data=pairs, caption=caption,
               notebook='analysis/3_Figure3/4_Fig3gh_MIP_vs_Aggregates.ipynb')
    plt.show()


agg_vs_mip('Metadata_conc_step', 'Blues', 'Fig3g',
           'Replicate correlation, MIP vs aggregates, coloured by concentration step',
           'Metadata_conc_step')

In [ ]:
agg_vs_mip('Metadata_pathway', LUT, 'Fig3h',
           'Replicate correlation, MIP vs aggregates, coloured by pathway',
           'Metadata_pathway')